# Patrón Creacional: Prototype

## Introducción
El patrón Prototype permite copiar objetos existentes sin que el código dependa de sus clases. Es útil cuando la creación de un objeto es costosa o compleja, o cuando se necesita clonar objetos configurados previamente.

## Objetivos
- Comprender el propósito y la implementación del patrón Prototype.
- Identificar cuándo es útil y cuándo evitarlo.
- Comparar la solución con y sin el patrón.

## Ejemplo de la vida real
**Contexto: App de Banco**
Supón que un banco tiene plantillas de contratos o productos financieros. Cuando un cliente solicita un producto, el sistema clona la plantilla y la personaliza. El patrón Prototype permite clonar objetos configurados sin depender de su clase concreta.

**¿Dónde se usa en proyectos reales?**
En sistemas de gestión de documentos, generación de contratos, creación de usuarios a partir de plantillas, etc.

## Sin patrón Prototype (forma errónea)
El código cliente debe crear manualmente nuevos objetos y copiar sus atributos. Esto puede ser propenso a errores y difícil de mantener si el objeto tiene muchos atributos.

In [1]:
class Documento:
    def __init__(self, texto):
        self.texto = texto
        self.revisar_ortografia()
        self.formatear()

    def revisar_ortografia(self):
        print('Revisando ortografía...')

    def formatear(self):
        print('Aplicando formato...')

doc1 = Documento('original')
doc2 = Documento(doc1.texto)
print(doc2.texto)

Revisando ortografía...
Aplicando formato...
Revisando ortografía...
Aplicando formato...
original


## Con patrón Prototype (forma correcta)
El cliente puede clonar objetos existentes fácilmente, sin preocuparse por los detalles internos de la clase. Esto facilita la reutilización y la personalización.

In [2]:
import copy
class Documento:
    def __init__(self, texto):
        self.texto = texto
    def clonar(self):
        return copy.deepcopy(self)

doc1 = Documento('original')

doc2 = doc1.clonar()
print(doc2.texto)
doc2.texto = 'copia modificada'
print(doc1.texto)
print(doc2.texto)


original
original
copia modificada


In [3]:
doc3 = doc1
print(doc3.texto)

print(doc1 is doc2)
print(doc1 is doc3)

doc3.texto = 'modificado a través de doc3'
print(doc1.texto)
print(doc3.texto)

original
False
True
modificado a través de doc3
modificado a través de doc3


## UML del patrón Prototype
```plantuml
@startuml
class Documento {
    + __init__(texto)
    + clonar()
}
@enduml
```

## Otro ejemplo de la vida real: Variantes de producto en un catálogo de e-commerce
**Contexto:** en tiendas como Shopify o Mercado Libre, un mismo producto (ej. una camiseta) tiene múltiples **variantes** (color, talla) que comparten nombre, categoría, descripción y especificaciones técnicas, pero difieren en color, talla, precio y stock. Crear cada variante repitiendo todos los campos compartidos es tedioso y propenso a inconsistencias (¿qué pasa si cambian la descripción y se les olvida actualizar una de las 12 variantes?).

### Sin patrón (forma errónea)
Cada variante se construye repitiendo manualmente los datos compartidos del producto base.

In [4]:
class Producto:
    def __init__(self, nombre, categoria, descripcion, especificaciones, color, talla, precio, stock):
        self.nombre = nombre
        self.categoria = categoria
        self.descripcion = descripcion
        self.especificaciones = especificaciones
        self.color = color
        self.talla = talla
        self.precio = precio
        self.stock = stock

# Los datos compartidos (nombre, categoria, descripcion, especificaciones) se repiten en cada variante
camiseta_roja_m = Producto(
    'Camiseta Deportiva', 'Ropa', 'Camiseta transpirable de secado rápido',
    {'material': 'poliéster', 'garantia_meses': 6},
    color='rojo', talla='M', precio=59900, stock=20
)
camiseta_azul_l = Producto(
    'Camiseta Deportiva', 'Ropa', 'Camiseta transpirable de secado rápido',
    {'material': 'poliéster', 'garantia_meses': 6},
    color='azul', talla='L', precio=59900, stock=15
)

### Con patrón (forma correcta)
Se define un producto base **una sola vez** y cada variante se obtiene clonándolo (`copy.deepcopy` evita que las variantes compartan el mismo diccionario de `especificaciones`) y ajustando solo los campos que cambian.

In [5]:
import copy

class Producto:
    def __init__(self, nombre, categoria, descripcion, especificaciones, color=None, talla=None, precio=None, stock=None):
        self.nombre = nombre
        self.categoria = categoria
        self.descripcion = descripcion
        self.especificaciones = especificaciones
        self.color = color
        self.talla = talla
        self.precio = precio
        self.stock = stock

    def clonar(self, **cambios):
        nuevo = copy.deepcopy(self)
        for atributo, valor in cambios.items():
            setattr(nuevo, atributo, valor)
        return nuevo

    def mostrar(self):
        print(f'{self.nombre} ({self.color}, talla {self.talla}) - ${self.precio} - stock: {self.stock}')


camiseta_base = Producto(
    'Camiseta Deportiva', 'Ropa', 'Camiseta transpirable de secado rápido',
    {'material': 'poliéster', 'garantia_meses': 6}
)

camiseta_roja_m = camiseta_base.clonar(color='rojo', talla='M', precio=59900, stock=20)
camiseta_azul_l = camiseta_base.clonar(color='azul', talla='L', precio=59900, stock=15)

camiseta_roja_m.mostrar()
camiseta_azul_l.mostrar()
print(camiseta_base.especificaciones is camiseta_roja_m.especificaciones)

Camiseta Deportiva (rojo, talla M) - $59900 - stock: 20
Camiseta Deportiva (azul, talla L) - $59900 - stock: 15
False


### UML del ejemplo de variantes de producto
```plantuml
@startuml
class Producto {
    + nombre
    + categoria
    + descripcion
    + especificaciones
    + color
    + talla
    + precio
    + stock
    + clonar(cambios)
    + mostrar()
}
@enduml
```

### ¿Dónde más se usa Prototype?
- **Catálogos e-commerce:** variantes de color/talla de un mismo producto (Shopify, Mercado Libre), como en este ejemplo.
- **Videojuegos:** clonar un enemigo o vehículo base (con sus stats, hitbox y animaciones ya configuradas) y solo ajustar posición o nivel, en vez de reconstruirlo desde cero en cada spawn.
- **Editores de diseño (Figma, PowerPoint):** "duplicar" un objeto conserva estilos, tamaño y efectos, y solo cambia lo que el usuario edite después.
- **Plantillas de documentos/contratos:** clonar una plantilla legal ya revisada por legal/compliance en vez de redactar cada contrato desde cero.
- **Configuración de infraestructura:** clonar una plantilla de máquina virtual o contenedor ya aprovisionada (con paquetes y permisos) para levantar una nueva instancia más rápido que reconstruirla.

**Ejercicio de reflexión:** ¿por qué `clonar()` usa `copy.deepcopy` en vez de simplemente crear un nuevo objeto con `Producto(**vars(self))`? Prueba a cambiar `especificaciones` de una variante sin `deepcopy` y observa qué le pasa al producto base.

## Actividad
Crea tu propio Prototype para clonar objetos de una clase personalizada (por ejemplo, usuario, producto, etc.).

---

## Explicación de conceptos clave
- **Clonación eficiente:** Prototype permite clonar objetos complejos sin depender de su clase concreta.
- **Desacoplamiento:** El cliente no necesita conocer los detalles de la clase para crear copias.
- **Aplicación en la vida real:** Útil en sistemas donde se reutilizan plantillas, contratos o configuraciones.

## Conclusión
El patrón Prototype es ideal para sistemas donde la creación de objetos es costosa o compleja, o cuando se requiere clonar objetos configurados. Facilita la reutilización y la personalización, y es común en aplicaciones bancarias, gestión de documentos y sistemas de plantillas.